# Datamine MODRES Process Example

This notebook demonstrates how to configure and run the **`modres`** process wrapper in `dmstudio`.

## Process Description

## MODRES

Evaluate an ore body model through one or more sets of perimeters defining an open pit, using full or partial block evaluation, and writes an output file of results.

This file may be read by the [TABRES](<tabres.md>) process to produce various reserve tabulations. An output model, containing a MINED field, may be produced. Multiple pits may be evaluated in a single run. Benches for evaluation need not match the levels in the model.

### Input and Output Model

The input model is in standard DATAMINE model file format, and may contain cells and sub-cells, in any order; the model does not have to be sorted on IJK to be processed by **MODRES**.

The fraction of the cell or sub-cell mined may be stored in the **MINED** field of the optional output model file, as a value in the range 0 to 1\.

The output model file will be an exact copy of the input file, with the **MINED** field added if it does not already exist in the input file. The output model file may be the same as the input model file (in-place updating) if the **MINED** field exists in the input model file.

### No Perimeter File

If no perimeter file is entered, the entire model is evaluated bench by bench and the results written to the &**RESULTS** file. Note that the process constructs a dummy perimeter in this case, and will therefore still record that 1 perimeter was read.

### Perimeter Conventions

If a perimeter file is specified, the perimeters must apply to benches. All perimeters must be specified clockwise; they must not be closed. The perimeter file is in the standard Datamine perimeter format. It must contain the numeric and explicit fields **XP** , **YP** , **ZP** , **PTN** and **PVALUE** , where **PTN** is the point number in the perimeter and PVALUE is a numeric perimeter identifier. Each perimeter must have a unique **PVALUE** identifier. All points for the same perimeter must be together in the file.

There are two forms of the perimeter file, specified by the value of the @PAIRS parameter.

### Bench Centre Perimeters

If @**PAIRS** =0 (default) then each perimeter is assumed to lie at the bench centre.

If @**ZVALUE** =0 (default) then the **ZP** field is ignored and the bench that each perimeter relates to is determined by its PVALUE. The perimeter identifier **PVALUE** must have the form _bbbbbb.nn_ where _bbbbbb_ is the bench number and _nn_ is an identifier on the bench. For example, two perimeters for bench 6 could have **PVALUE** s of 6.0 and 6.1. There can be multiple perimeters on one bench, as long as each is separately identified in this way. Note that Bench 1 refers to the uppermost level of model cells.

If @**ZVALUE** =1 then the bench to which each perimeter belongs will be identified by its **ZP** value, and not by its **PVALUE**. Normally, all points in a single perimeter would have the same **ZP** value, but if the **ZP** values vary, then the bench number is derived from the **ZP** value of the last point in the perimeter string.

The order of perimeters in the file does not matter, but within one run of **MODRES** , perimeters are _ADDITIVE_ , and therefore should not overlap. However, between different runs of **MODRES** , perimeters are _INCREMENTAL_ on what has been previously mined (although additive on each other) and therefore should overlap.

### Bench Bottom and Top Perimeters

If parameter @**PAIRS** =1, then benches are no longer defined by the model cells in Z; they are defined by consecutive pairs of perimeters, one for the bottom of the bench and one for the top. The elevation of each perimeter is defined by the **ZP** value, and the bench number by the _bbbbbb_ component of the **PVALUE** field.

For example, two consecutive perimeters in the file could have **PVALUE** s of 3.00 and 3.01, with ZP values of 275 and 282. These would define bench 3 as lying between elevations 275 and 282, irrespective of the model dimensions. Note that, as before, each perimeter must have a different **PVALUE**.

The bench bottom and top perimeters may be different, allowing a pit to be defined in as much detail as required. The perimeter defining the top of one bench to be mined may be different from the perimeter defining the bottom of the next bench up. Thus **MODRES** may be used to evaluate highly detailed open pits, which can include haul roads or other features. Where bench top and bottom perimeters are different, **MODRES** interpolates intermediate perimeters as required, allowing accurate evaluation.

### Evaluation Parameters

Evaluation parameters are specified interactively. All results may be classified by rocktype, either as selected values of a rocktype field, or by classing these field values as one of a set of **ORE, WSTE, AIR, S1, S2, S3, S4, S5, S6, S7**. The S1 - S7 classifications are designed for stockpile material, but they may also be used as alternate ore or waste classifications if required.

When classifying reserves by selected values of rocktype, unnamed rocktypes will be assigned to the final rocktype in the list. It is good practice to add a dummy rocktype to the list (e.g. XXXX or 9999) so that the presence of unnamed rocktypes will be easily detected.

When classifying reserves by **ORE, WSTE** (and so on) unnamed rocktypes are assigned to **WSTE**.

The results may also be classified by grade intervals for a selected main grade field. These grade intervals do not have to be uniform.

The full possible hierarchic classification of results is:-

Bench \- Perimeter - Rocktype - Grade interval.

All numeric fields which are not standard model or perimeter fields, and which are not chosen as the rocktype or main grade field, are also evaluated automatically. See the Errors section for exceptions to this.

### Balancing Volumes

**MODRES** also produces, for each perimeter, a balancing volume, which is the difference between the volume within the perimeter (computed accurately) and the sum of volumes of the cells and sub-cells mined within the perimeter.

If the perimeters lie wholly within the modelled region, for partial block evaluation (default) these balancing volumes are caused by small arithmetic inaccuracies; for full block evaluation (parameter @**FULLCELL** =1) they will be both positive and negative, caused by the difference in volume between the sum of all cells and sub-cells whose center lies within the perimeters and the actual volume calculated as bench height * the perimeter area.

If the perimeters cover volumes greater than those modelled (either because the perimeters go outside the model region, or because cells or sub-cells within the model are absent) then the balancing volumes represent as accurately as possible the volume contained within the perimeter which is mined but not modelled. Modelled volumes which were mined at evaluation time are excluded from these balances.

Small negative balancing volumes can occur, particularly if full cell evaluation is being used. However, large negative balances are an indication of errors in either the input model or in the perimeters. A model that contains duplicate or overlapping cells will cause negative balances, because the total volume of the cells is greater than the space they occupy. The [PROMOD](<promod.md>) command can be used to check for such errors.

Perimeters that contain crossovers (i.e. "figure eight" shapes) can also cause negative balances. Such perimeters contain both clockwise and anti-clockwise sections. If the enclosed area of the clockwise sections is greater than that of the anti-clockwise sections, then the perimeter is treated as clockwise, but the anti-clockwise sections will be reported as negative balancing volumes. If the reverse is true, then the perimeter is treated as anti-clockwise or malformed and a fatal error occurs. See Errors below.

### Densities

In computation of tonnages from volumes, any **DENSITY** field within the input model is used. If the **DENSITY** field exists, then unmodelled regions will use the default value of the **DENSITY** field. If the **DENSITY** field does not exist or if its value is absent, then an overall density may be supplied by use of the @**DENSITY** parameter. If this was not supplied, then the density used is 1.0.

### Passes through the Model File

The process reads in as many perimeters from the optional input perimeter file as possible (maximum 100, or the number of points to occupy half the virtual array space. This space is 2,000,000 words (1,000,000 points).

The model is then read (sub-)cell by (sub-)cell, and each evaluated in turn against all the perimeters.

Therefore, if the number of perimeters is less than 100 and all perimeter points fit in half the virtual array space, then the model is read just once to perform all evaluations (a single pass).
Otherwise, the model is read once for each set of perimeters (multiple passes). If multiple passes are required, and incremental perimeters are entered (i.e. they overlap on a bench) then an output model must be specified, as it is used to store the intermediate results from each pass.

### Input Files:

* **in** (Block Model):
  Model file for evaluation. Must contain at least the fields **XC, YC, ZC, XINC, YINC,

## ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ, IJK**.

  Required=Yes

* **perimin** (String):
  Optional input perimeter file. Must contain at least the fields **XP, YP, ZP, PTN,
  PVALUE**. The **PVALUE** field contains the bench number as the integer part: e.g. 3.0
  for first perim on bench 3, 3.1 for second etc. If the **PAIRS** option is set, then the
  ZP value must contain the bench bottom elevation in one perimeter, then the bench top in
  the next. All points for one perimeter must be together. Perimeters must be clockwise,
  unclosed.
  Required=No

### Output Files:

* **results** (Undefined):
  The output results file, in a format suitable for input into the **TABRES** process.
  Required=Yes

* **out** (Block Model):
  Optional output model file. This may be the same as the input, if the **MINED** field
  exists in the input file. The **MINED** field will be created in the output file if it
  does not exist.
  Required=No

### Fields:

### Parameters:

* **density**:
  Set required density value. This will only be used if there is no **DENSITY** field in
  the input model. If there is no **DENSITY** field, and no **DENSITY** parameter, then a
  value of 1.0 is used.
  Range=0.000001,+
  Values=Undefined
  Default=1
  Required=Yes

* **zvalue**:

* **Options** (1: For single perimeters at bench centres, take the Z value from the **ZP**):
  field instead of using the level number in the **PVALUE** field.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **pairs**:

* **Options** (1: Use pairs of perimeters to define bench bottoms and tops, as defined by ZP):
  field.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **fullcell**:

* **Options** (1: whole cell evaluation in place of partial cell evaluation.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:

* **Options** (1: Show a line for each cell evaluated in each perimeter.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('modres')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute modres
print("Running modres...")
dm_cmd.modres(
    in_i='t_assays',  # required
    results_o=['t_modres_out'],  # required
    density_p='required_val',  # required
    # perimin_i='optional',  # optional
    # out_o='t_modres_out',  # optional
    # zvalue_p=0,  # optional
    # pairs_p=0,  # optional
    # fullcell_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("modres execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_modres_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")